In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

#aqui estou utilizando as funções de checagem disponibilizadas no material da aula de arquitetura medalhão para a minha solução pois ela é bastante completa e eficiente, além de ter sido disponibilizada diretamente por um canal oficial da visagio, o que valida ainda mais a qualidade das funções

#o que elas fazem, resumidamente, é armazenar as métricas numa lista global dq_results usando objetos Row, e depois retornar uma tabela com os resultados. cada função usa métricas únicas para cada tabela, mas a função de checagem é a mesma para todas as tabelas, então basta alterar o nome da tabela e a função de checagem para cada uma
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from itertools import chain

#criacao do dataframe de consulta da tabela bronze.tb_movies_info
df_info = spark.table("bronze.tb_movies_info")    

#renomeação do titulo das colunas 
df_info = (df_info
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

#tratamento da deduplicação mantendo a versão mais recente do filme com base na data de ingestão
deduplicacao = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc()) #criação da janela de deduplicação que ordena os dados por data de ingestão

#atualiza o df com a deduplicação dos dados mantendo a versão mais recente do filme com base na data de ingestão 
df_info = (df_info
    .withColumn("row_number", F.row_number().over(deduplicacao))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .filter(F.col("id_filme").isNotNull())          #filtro de verificacao de ids nulos
)

#traducao das colunas
status_filme = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado"
}
#criacao de um mapa de tradução dos status usando o status_filme 
translate_status_map = F.create_map([F.lit(x) for x in chain.from_iterable(status_filme.items())])

#limpeza dos status dos filmes
df_info = (df_info
    #remocao dos hifens e espaços vazios 
    .withColumn("status_limpo", F.lower(F.trim(F.regexp_replace(F.col("status_filme"), r"-", ""))))
    # se bater com o mapa de traducoes, traduzir, se não, retornar "Não Informado"
    .withColumn("status_filme", F.coalesce(translate_status_map[F.col("status_limpo")], F.lit("Não Informado")))
    .drop("status_limpo")
)

#limpeza de minusculos (para que todos os titulos estejam em maiusculo na coluna 'titulo')
df_info = df_info.withColumn("titulo", F.initcap(F.trim(F.col("titulo"))))

#lista de padronização de datas
date_format = ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy", "MM-dd-yyyy"]

#tratamento de datas multi-formato 
#a atualização do dataframe usa a lista date_fmt para tratar os diferentes formatos de datas da tabela movies_info aplicada num loop for, usando o coalesce para retornar a primeira data válida da lista, jogando para o campo ano_lancamento (nova coluna) 
df_info = (df_info
    .withColumn("data_lancamento", F.coalesce(*[F.try_to_date(F.col("data_lancamento"), fmt) for fmt in date_format]))
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

#checks de qualidade de dados antes da gravação
dq_check_unique("tb_info_filmes", "unicidade_id_filme", df_info, ["id_filme"])
dq_check("tb_info_filmes", "id_filme_nao_nulo", df_info, F.col("id_filme").isNotNull())
dq_check("tb_info_filmes", "status_traduzido", df_info, F.col("status_filme") != "Não Informado")
dq_check("tb_info_filmes", "data_lancamento_valida", df_info, F.col("data_lancamento").isNotNull())
dq_check("tb_info_filmes", "ano_lancamento_extraido", df_info, F.col("ano_lancamento").isNotNull())

#criacao da tabela usando mode overwrite para manter a unicidade da tabela
df_info.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")

display(spark.table("silver.tb_info_filmes").limit(10))                                       

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from itertools import chain

#criacao do dataframe de consulta da tabela bronze.tb_movies_financials e da bronze.tb_cotacao_dolar
df_financials = spark.table("bronze.tb_movies_financials")  
df_cotacao = spark.table("bronze.tb_cotacao_dolar")

#calculo da cotacao atual buscando na coluna a data da cotacao mais recente (dos ultimos 7 dias)
cotacao_atual = (df_cotacao
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)

#deduplicacao (em funcao da ingestion datetime) e renomeacao das colunas no dataframe
deduplicacao = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_financials = (df_financials
    .withColumn("row_number", F.row_number().over(deduplicacao))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .filter(F.col("id").isNotNull())
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
)

def tratamento_moeda(coluna):
    #1o passo: remover caracteres especiais
    #when/otherwise funciona como uma estrutua condicional para retornar um valor ou outro (NULL ou o valor original, neste caso)
    result = F.when(F.col(coluna).isin("Unknown", "", "Não Informado"), F.lit(None)).otherwise(F.col(coluna))   
    
    #2o passo: remover simbolos de moeda e pontuacoes de milhar
    result = F.regexp_replace(result, r"[^\d\.]", "")  #faz a busca procurando o que NÃO FOR algorismos de 0 a 9 ou ponto final (ignorando simbolos e virgulas)

    #3o passo: converter para decimal
    #primeiro valor = precisao (numero maximo de digitos/margem de segurança); segundo valor = escala (numero de casas decimais/)
    result = F.when(result == "", F.lit(None)).otherwise(result).cast("decimal(15,2)")    

    #4o passo: tratar valores zerados como ausentes
    result = F.when(result <= 0, F.lit(None)).otherwise(result)

    return result

#aplicaçao do tratamento da tabela no dataframe
df_financials = (df_financials
    .withColumn("orcamento_usd", tratamento_moeda("orcamento_usd"))
    .withColumn("receita_usd", tratamento_moeda("receita_usd"))
    )

#calculo da receita em reais, lucro e margem de lucro percentual
df_financials = (df_financials
    #conversão de usd para brl - passando o valor da cotacao_atual no mesmo formato das colunas da tabela financials pós tratamento (decimal(15,2))
    .withColumn("receita_brl", F.col("receita_usd") * F.lit(cotacao_atual).cast("decimal(15,2)"))      #F.lit(cotacao_atual) neste caso ajuda a manter o padrao
    .withColumn("orcamento_brl", F.col("orcamento_usd") * F.lit(cotacao_atual).cast("decimal(15,2)"))  #numerico das linhas de ambas as colunas

    #calculo do lucro USD/BRL
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(15,2)"))   #lucro em dolares
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(15,2)"))   #lucro em reais 
    
    #margem de lucro percentual - guarda explícita contra divisão por zero e valores ausentes (NULL)
    .withColumn("margem_lucro_percentual_usd", F.when(F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") != 0), (F.col("lucro_usd") / F.col("orcamento_usd") * 100).cast("decimal(15,2)")).otherwise(F.lit(None)))
    .withColumn("margem_lucro_percentual_brl", F.when(F.col("orcamento_brl").isNotNull() & (F.col("orcamento_brl") != 0), (F.col("lucro_brl") / F.col("orcamento_brl") * 100).cast("decimal(15,2)")).otherwise(F.lit(None))) 

    #OBS: adicionei o cast(decimal) novamente para garantir que as operações matemáticas não reajustem os valores das colunas, mantendo o padrao original 15,2

    )

#checks de qualidade de dados antes da gravação
dq_check_unique("tb_financeiro_filmes", "unicidade_id_filme", df_financials, ["id_filme"])
dq_check("tb_financeiro_filmes", "id_filme_nao_nulo", df_financials, F.col("id_filme").isNotNull())
dq_check("tb_financeiro_filmes", "orcamento_positivo", df_financials, (F.col("orcamento_usd").isNull()) | (F.col("orcamento_usd") > 0))
dq_check("tb_financeiro_filmes", "receita_positiva", df_financials, (F.col("receita_usd").isNull()) | (F.col("receita_usd") > 0))
dq_check("tb_financeiro_filmes", "margem_calculada", df_financials, (F.col("orcamento_usd").isNull()) | (F.col("margem_lucro_percentual_usd").isNotNull()))

df_financials.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")

display(spark.table("silver.tb_financeiro_filmes").limit(50))

#COMENTARIO: o teste relacionado ao calculo de margem deu falha por conta de valores NULL que compoem o calculo (receita e orçamento) - por isso o número de 6515 falhas, o que representa 86,2% de taxa de sucesso, que na verdade são 100% (pois todas as margens que possuem ambos os valores componentes do calculos preenchidos corretamente foram calculadas corretamente)


In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_metrics = spark.table("bronze.tb_movies_metrics")

#deduplicacao com base na ingestion datetime e tradução dos nomes das colunas
deduplicacao = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_metrics = (df_metrics
    .withColumn("row_number", F.row_number().over(deduplicacao))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .filter(F.col("id").isNotNull()) 
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("popularity","popularidade")
    .withColumnRenamed("vote_average","nota_media_tmdb")
    .withColumnRenamed("vote_count","qtd_votos_tmdb")
    .withColumnRenamed("averageRating","nota_media_imdb")
    .withColumnRenamed("numVotes","qtd_votos_imdb")
)

#limpeza/higiene dos dados da coluna popularidade, garantindo que valores decimais separados com ',' sejam substutuidos por '.' evitando remoção 
df_metrics = df_metrics.withColumn(
    "popularidade", 
    F.regexp_replace(F.col("popularidade"), r",", ".")
)

#casting/conversao de tipo - notas: decimal, votos: integer (nao existem 4,5 votos, por exemplo)
df_metrics = (df_metrics
    .withColumn("popularidade", F.col("popularidade").try_cast("decimal(10,2)"))
    .withColumn("nota_media_tmdb", F.col("nota_media_tmdb").try_cast("decimal(10,1)"))
    .withColumn("qtd_votos_tmdb", F.col("qtd_votos_tmdb").try_cast("integer")) 
    .withColumn("nota_media_imdb", F.col("nota_media_imdb").try_cast("decimal(10,1)"))
    .withColumn("qtd_votos_imdb", F.col("qtd_votos_imdb").try_cast("integer"))
)

df_metrics = (df_metrics
    #neste bloco de update do dataframe, usando a condicional when/otherwise, o programa vai atualizar o dataframe obedecendo às regras de limitação do intervalo de notas (0 a 10) e de tratamento de numeros negativos

    #restricao do valor das notas ao intervalo entre 0 e 10
    .withColumn("nota_media_tmdb", F.when((F.col("nota_media_tmdb") < 0) | (F.col("nota_media_tmdb") > 10), F.lit(None)).otherwise(F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when((F.col("nota_media_imdb") < 0) | (F.col("nota_media_imdb") > 10), F.lit(None)).otherwise(F.col("nota_media_imdb")))
    
    #bloqueio de popularidade negativa
    .withColumn("popularidade", F.when(F.col("popularidade") < 0, F.lit(None)).otherwise(F.col("popularidade")))

    #bloqueio de contagem de votos negativa - tmdb e imdb
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") < 0, F.lit(None)).otherwise(F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") < 0, F.lit(None)).otherwise(F.col("qtd_votos_imdb")))
)


#checks de qualidade de dados antes da gravação
dq_check_unique("tb_metricas_engajamento", "unicidade_id_filme", df_metrics, ["id_filme"])
dq_check("tb_metricas_engajamento", "id_filme_nao_nulo", df_metrics, F.col("id_filme").isNotNull())
dq_check("tb_metricas_engajamento", "notas_tmdb_validas", df_metrics, (F.col("nota_media_tmdb").isNull()) | ((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10)))
dq_check("tb_metricas_engajamento", "notas_imdb_validas", df_metrics, (F.col("nota_media_imdb").isNull()) | ((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10)))
dq_check("tb_metricas_engajamento", "popularidade_nao_negativa", df_metrics, (F.col("popularidade").isNull()) | (F.col("popularidade") >= 0))

#usei overwriteSchema para reescrever o valor de popularidade, garantindo que o casting float -> decimal(10,1) seja aplicado corretamente (decidi usar UMA casa decimal para as notas no casting pois é o formato mais comum na internet para dar notas para filmes)
df_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_metricas_engajamento")

display(spark.table("silver.tb_metricas_engajamento").limit(200))

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_reviews = spark.table("bronze.tb_movies_reviews")

#deduplicacao com base na ingestion datetime e tradução dos nomes das colunas 
deduplicacao = Window.partitionBy("id", "nome", "nota", "comentario").orderBy(F.col("ingestion_datetime").desc())
df_reviews = (df_reviews
    .withColumn("row_number", F.row_number().over(deduplicacao))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .filter(F.col("id").isNotNull()) 
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("nome","nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario","comentario_usuario")                         
)

#tratamento de notas (limitação de intervalo - 0 a 10) e casting para decimal 
df_reviews = (df_reviews
    .withColumn("nota_usuario", F.col("nota_usuario").try_cast("decimal(10,1)"))   #casting
    #restricao do valor das notas ao intervalo entre 0 e 10 usando when/otherwise como estrutura condicional
    .withColumn("nota_usuario", F.when((F.col("nota_usuario") < 0) | (F.col("nota_usuario") > 10), F.lit(None)).otherwise(F.col("nota_usuario")))  
)

#adição de texto generico "Sem comentário" para espaços vazios na coluna comentario_usuario
df_reviews = df_reviews.withColumn("comentario_usuario", 
    F.when(
        F.col("comentario_usuario").isNull() | (F.trim(F.col("comentario_usuario")) == ""), 
        F.lit("Sem comentário")
    ).otherwise(F.col("comentario_usuario"))
)

#checks de qualidade de dados antes da gravação
dq_check("tb_avaliacoes_usuarios", "id_filme_nao_nulo", df_reviews, F.col("id_filme").isNotNull())
dq_check("tb_avaliacoes_usuarios", "nota_usuario_valida", df_reviews, (F.col("nota_usuario").isNull()) | ((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10)))
dq_check("tb_avaliacoes_usuarios", "comentario_preenchido", df_reviews, F.col("comentario_usuario").isNotNull())
dq_check("tb_avaliacoes_usuarios", "nome_usuario_nao_nulo", df_reviews, F.col("nome_usuario").isNotNull())

df_reviews.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")

display(spark.table("silver.tb_avaliacoes_usuarios").limit(100))

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_credits = spark.table("bronze.tb_credits_and_tags")

#tratamento de delimitadores (substituindo ';' e '|' por ',')
df_genres = df_credits.withColumn(
    "genres_normalizado", 
    F.regexp_replace(F.col("genres"), r"[;|]", ",")
) 

#split+explode: expansão da coluna e criação de 'nome_genero'
df_genres = (df_genres
    .withColumn("genres_array", F.split(F.col("genres_normalizado"), ","))
    .withColumn("nome_genero", F.explode(F.col("genres_array")))
)

#higienização dos dados na coluna correta (nome_genero)
df_genres = (df_genres
    # Remove espaços vazios nas pontas e limpa resíduos de aspas ou colchetes
    .withColumn("nome_genero", F.trim(F.regexp_replace(F.col("nome_genero"), r'[\"\[\]\']', "")))
    # Primeira letra em maiúsculo para manter um padrão visual limpo
    .withColumn("nome_genero", F.initcap(F.col("nome_genero")))
)

#tratamento de qualidade dos dados
df_genres = (df_genres
    .filter(F.col("id").isNotNull())
    .filter(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .filter(~F.col("nome_genero").rlike(r"^\d+$")) # descarta se for puramente numérico
    .filter(~F.col("nome_genero").rlike(r"^/.*\.jpg$")) # descarta paths de imagem
)

#whitelist de gêneros válidos (padrão TMDB) para eliminar frases, taglines e lixo que vieram misturados na coluna genres
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "Tv Movie", "Thriller", "War", "Western"
]

df_genres = df_genres.filter(F.col("nome_genero").isin(generos_validos))

#seleção final com as colunas id e genero
df_genres = (df_genres
    .select(
        F.col("id").alias("id_filme"), 
        F.col("nome_genero")
    )
    .dropDuplicates()
)

#checks de qualidade de dados antes da gravação
dq_check("tb_generos", "id_filme_nao_nulo", df_genres, F.col("id_filme").isNotNull())
dq_check("tb_generos", "nome_genero_nao_vazio", df_genres, (F.col("nome_genero").isNotNull()) & (F.trim(F.col("nome_genero")) != ""))
dq_check("tb_generos", "genero_na_whitelist", df_genres, F.col("nome_genero").isin(generos_validos))

df_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_generos")

display(spark.table("silver.tb_generos"))

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_credits = spark.table("bronze.tb_credits_and_tags")

def processar_entidade(df, coluna_origem, tipo_entidade):
    df_proc = df.withColumn(
        "normalizado", 
        F.regexp_replace(F.col(coluna_origem), ";", ",")
    )
    
    # Split + Explode
    df_proc = (df_proc
        .withColumn("array_itens", F.split(F.col("normalizado"), ","))
        .withColumn("nome_limpo", F.explode(F.col("array_itens")))
    )
    
    #higienização de texto (aspas, colchetes, espaços e capitalização)
    df_proc = df_proc.withColumn(
        "nome_limpo", 
        F.initcap(F.trim(F.regexp_replace(F.col("nome_limpo"), r'[\"\[\]\']', "")))
    )
    
    #filtros de qualidade (remove nulos, vazios e números isolados por column shift)
    df_proc = (df_proc
        .filter(F.col("id").isNotNull())
        .filter(F.col("nome_limpo").isNotNull() & (F.col("nome_limpo") != ""))
        .filter(~F.col("nome_limpo").rlike(r"^\d+$"))
    )
    
    #seleção final padronizada com a coluna de tipo atribuída via parâmetro
    return df_proc.select(
        F.col("id").alias("id_filme"),
        F.col("nome_limpo").alias("nome_entidade"),
        F.lit(tipo_entidade).alias("tipo_entidade")
    )

#processamento individual de cada uma das 4 entidades 
df_cast = processar_entidade(df_credits, "cast", "Ator")
df_directors = processar_entidade(df_credits, "directors", "Diretor")
df_writers = processar_entidade(df_credits, "writers", "Roteirista")
df_companies = processar_entidade(df_credits, "production_companies", "Produtora")

#união de todas as entidades num único DataFrame consolidado
df_pessoas_empresas = df_cast.unionByName(df_directors).unionByName(df_writers).unionByName(df_companies)

#deduplicação final para garantir a unicidade da tabela
df_pessoas_empresas = df_pessoas_empresas.dropDuplicates()

#checks de qualidade de dados antes da gravação
dq_check("tb_pessoas_empresas", "id_filme_nao_nulo", df_pessoas_empresas, F.col("id_filme").isNotNull())
dq_check("tb_pessoas_empresas", "nome_entidade_nao_vazio", df_pessoas_empresas, (F.col("nome_entidade").isNotNull()) & (F.trim(F.col("nome_entidade")) != ""))
dq_check("tb_pessoas_empresas", "tipo_entidade_valido", df_pessoas_empresas, F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista", "Produtora"]))

df_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_pessoas_empresas")

display(spark.table("silver.tb_pessoas_empresas").limit(1000))


In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import warnings

#supressão do warning de Window sem partição (esperado para forward fill em série temporal única)
warnings.filterwarnings('ignore', category=UserWarning, module='pyspark.sql.connect.expressions')

df_cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")

#conversao de data string -> date
df_cotacao = df_cotacao_bronze.select(
    F.to_date(F.col("dataHoraCotacao")).alias("data_cotacao_real"),
    F.col("cotacaoCompra").cast("decimal(10,4)").alias("valor_cotacao")
).dropDuplicates(["data_cotacao_real"])

#dscobrir o intervalo de datas (min e max) presente na base
limites = df_cotacao.agg(
    F.min("data_cotacao_real").alias("min_data"),
).collect()[0]

data_inicial = limites["min_data"]

#criacao de um calendario (registro) de datas continuo usando sql para gerar uma sequencia de datas e explodir 
df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_inicial}'), current_date(), interval 1 day)) as data_referencia
""")

#aqui usei left join para agrupar os dados do calendario criado com os dados da API,
#fins de semana ficarao com os dados marcados como NULL
df_merged = df_calendario.join(
    df_cotacao, 
    df_calendario.data_referencia == df_cotacao.data_cotacao_real, 
    "left"
)

#forward fill para preencher os ultimos dias sem cotacao com os ultimos valores nao nulos
janela_ffill = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)
#a função 'last' com ignorenulls=True vai puxar a última cotação não-nula que encontrar para trás.
df_silver_cotacao = (df_merged
    .withColumn("cotacao_usd_brl", F.last(F.col("valor_cotacao"), ignorenulls=True).over(janela_ffill))
    .select(
        F.col("data_referencia").alias("data_cotacao"),
        F.col("cotacao_usd_brl")
    )
)

#checks de qualidade de dados antes da gravação
dq_check_unique("tb_cotacao_dolar", "unicidade_data_cotacao", df_silver_cotacao, ["data_cotacao"])
dq_check("tb_cotacao_dolar", "data_cotacao_nao_nula", df_silver_cotacao, F.col("data_cotacao").isNotNull())
dq_check("tb_cotacao_dolar", "cotacao_preenchida", df_silver_cotacao, F.col("cotacao_usd_brl").isNotNull())
dq_check("tb_cotacao_dolar", "cotacao_positiva", df_silver_cotacao, F.col("cotacao_usd_brl") > 0)

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_cotacao_dolar")

display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao").limit(15))




In [0]:
#adicionei esta celula de relatorio de qualidade juntamente ao Genie, para verificarmos e agruparmos os resultados de todas as checagens de qualidade do notebook

#novamente chamo atenção quanto à questão da tabela silver.tb_financeiro_filmes, onde o teste de calculo de margem deu 6515 falhas, mas que se deve justamente à aplicação correta das regras de negócio quanto ao tratamento de dados não aceitáveis, classificando como NULL valores de receita fora de padrão ou corrompidos

if dq_results:
    df_dq_report = spark.createDataFrame(dq_results)
    
    # Ordenar por tabela e check
    df_dq_report = df_dq_report.orderBy("table_name", "check_name")
    
    print("\n" + "="*80)
    print(" RELATÓRIO DE QUALIDADE DE DADOS - CAMADA SILVER")
    print("="*80 + "\n")
    
    display(df_dq_report)
    
    # Sumário executivo
    total_checks = df_dq_report.count()
    passed_checks = df_dq_report.filter(F.col("passed") == True).count()
    failed_checks = total_checks - passed_checks
    
    print(f"\nSUMÁRIO DE CHECAGEM:")
    print(f"   Total de checks: {total_checks}")
    print(f"   Passou: {passed_checks}")
    print(f"   Falhou: {failed_checks}")
    print(f"   Taxa de sucesso: {(passed_checks/total_checks)*100:.1f}%\n")
else:
    print("Nenhum check de qualidade foi executado ainda. Execute as células de tratamento primeiro.")